In [22]:
import cobra
import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *

In [23]:
compartments_ = {'c': 'cytosolic',  'l': 'lysosomal', 'm': 'mitochondrial', 'r': 'endoplasmic reticulum', 
                'e': 'extracellular space', 'x': 'peroxisomal', 'n': 'nuclear', 'g': 'golgi apparatus',
                'i': 'inner mitochondrial compartment', 'pm': 'plasma membrane'}

In [24]:
human_model = cobra.io.load_matlab_model(root_path + 'MammalianSecretoryRecon/MODELS/RECON2_2.mat')
# human_model_2 = cobra.io.load_json_model(local_data_path + 'raw/RECON3D.json')

# two genes in recon2_2 have HGNC:HGNC:### rather than HGNC:###, the following code corrects that issue

# since the two genes are involved in the same three reactions, simply need to rewrite these three reactions
# rather than looping through
genes_to_duplicate = [gene.id for gene in human_model.genes if gene.id.count(':') > 1]
g0, g1 = genes_to_duplicate[0], genes_to_duplicate[1]  
g_i = human_model.genes.get_by_id(g0)
r_i = list(g_i.reactions)
g_c = human_model.genes.get_by_id(g0[5:])

# the following line of code gets rid of both genes in genes to duplicate since they are incolved in the same
# reations
human_model.remove_reactions(r_i, remove_orphans=True)
for r in r_i:
    r0 = r.gene_reaction_rule.replace(g0, g0[5:])
    r.gene_reaction_rule = r0.replace(g1, g1[5:])



In [25]:
all_metabolites = [m.id for m in human_model.metabolites]
all_reactions = list(human_model.reactions)

compartments = ['n']
for nucleotide in ['a', 'c', 'u', 'g']:
    for phosphate in ['dp', 'mp']:
        for compartment in compartments:
            metabolite_id = nucleotide + phosphate + '[' + compartment + ']'
            
            # metabolite exists
            if metabolite_id not in all_metabolites: # assume in 'C'
                met_ = human_model.metabolites.get_by_id(metabolite_id.replace('[' + compartment + ']', '[c]')).copy()
                met_.compartment = compartment
                met_.id = met_.id.replace('[c]', '[' + compartment + ']')
                human_model.add_metabolites([met_])
            
            # transport from cytosol exists
            met_1 = human_model.metabolites.get_by_id(metabolite_id)
            met_2 = human_model.metabolites.get_by_id(metabolite_id.replace('[' + compartment + ']', '[c]'))
            r_ = [r for r in all_reactions if met_1 in r.metabolites.keys() and met_2 in r.metabolites.keys()]
            if len(r_) == 0:
                # add transport
                m_name = met_2.id.replace('[c]', '').upper()
                transport = cobra.Reaction(m_name + 't' + compartment)
                transport.name = m_name + ' ' + compartments_[compartment] + ' transport'
                transport.add_metabolites({met_2: -1, met_1: 1})
                transport.lower_bound = -1000
                human_model.add_reactions([transport])
            elif len(r_) == 1:
                r_ = r_[0]
                if not r_.reversibility: # make all exchanges reversible
                    r_.lower_bound = -1000
            else:
                reactants, products = list(), list()
                for r in r_:
                    reactants += r.reactants
                    products += r.products
                if not((met_1 in reactants and met_2 in products) or (met_2 in reactants and met_1 in products)):
                    raise ValueError('No reversibility')

In [26]:
compartments = ['n', 'm', 'x', 'r']
amino_acids = ['ala_L', 'arg_L', 'asn_L', 'asp_L', 'cys_L', 'glu_L', 'gln_L', 'gly', 'his_L', 'ile_L', 'leu_L', 
              'lys_L', 'met_L', 'phe_L', 'pro_L', 'ser_L', 'thr_L', 'trp_L', 'tyr_L', 'val_L']
for amino_acid in amino_acids:
        for compartment in compartments:
            metabolite_id = amino_acid + '[' + compartment + ']'
            
            # metabolite exists
            if metabolite_id not in all_metabolites: # assume in 'C'
                met_ = human_model.metabolites.get_by_id(metabolite_id.replace('[' + compartment + ']', '[c]')).copy()
                met_.compartment = compartment
                met_.id = met_.id.replace('[c]', '[' + compartment + ']')
                human_model.add_metabolites([met_])
            
            # transport from cytosol exists
            met_1 = human_model.metabolites.get_by_id(metabolite_id)
            met_2 = human_model.metabolites.get_by_id(metabolite_id.replace('[' + compartment + ']', '[c]'))
            r_ = [r for r in all_reactions if met_1 in r.metabolites.keys() and met_2 in r.metabolites.keys()]
            if len(r_) == 0:
                # add transport
                m_name = met_2.id.replace('[c]', '').upper()
                transport = cobra.Reaction(m_name + 't' + compartment)
                transport.name = m_name + ' ' + compartments_[compartment] + ' transport'
                transport.add_metabolites({met_2: -1, met_1: 1})
                transport.lower_bound = -1000
                human_model.add_reactions([transport])
            elif len(r_) == 1:
                r_ = r_[0]
                if not r_.reversibility: # make all exchanges reversible
                    r_.lower_bound = -1000
            else:
                reactants, products = list(), list()
                for r in r_:
                    reactants += r.reactants
                    products += r.products
                if not((met_1 in reactants and met_2 in products) or (met_2 in reactants and met_1 in products)):
                    raise ValueError('No reversibility')

In [27]:
cobra.io.save_json_model(human_model, local_data_path + 'processed/corrected_recon2_2.json')